In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

In [2]:
class CustomGridEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=50):
        super().__init__()
        self.size = 3
        self.max_steps = max_steps
        self.action_space = spaces.Discrete(4)
        self.start_position = np.array([0, 0], dtype=np.int32)
        self.goal_position = np.array([2, 2], dtype=np.int32)
        self.hole_position = np.array([2, 1], dtype=np.int32)
        self.action_to_direction = {
            0: np.array([0, 1], dtype=np.int32),
            1: np.array([-1, 0], dtype=np.int32),
            2: np.array([0, -1], dtype=np.int32),
            3: np.array([1, 0], dtype=np.int32),
        }
        self.observation_space = spaces.Dict({
            "agent": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
            "target": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
        })
        self.agent_position = self.start_position.copy()
        self.steps = 0

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None):
        super().reset(seed=seed)
        self.agent_position = self.start_position.copy()
        self.steps = 0
        return self._get_obs(), self._get_info()

    def _get_obs(self):
        return {
            "agent": self.agent_position.copy(),
            "target": self.goal_position.copy(),
        }

    def _get_info(self):
        return {"distance": int(np.abs(self.agent_position - self.goal_position).sum())}

    def step(self, action):
        direction = self.action_to_direction[int(action)]
        self.agent_position = np.clip(
            self.agent_position + direction, 0, self.size - 1
        ).astype(np.int32)
        self.steps += 1

        reached_goal = bool(np.array_equal(self.agent_position, self.goal_position))
        fell_in_hole = bool(np.array_equal(self.agent_position, self.hole_position))

        if reached_goal:
            reward = 10.0
        elif fell_in_hole:
            reward = -10.0
        else:
            reward = -0.1

        terminated = reached_goal or fell_in_hole
        truncated = self.steps >= self.max_steps and not terminated
        return self._get_obs(), reward, terminated, truncated, self._get_info()

    def render(self):
        grid = np.full((self.size, self.size), " ")
        grid[tuple(self.hole_position)] = "H"
        grid[tuple(self.goal_position)] = "G"
        grid[tuple(self.agent_position)] = "A"
        print("+---" * self.size + "+")
        for row in grid:
            print("|" + "|".join(f" {cell} " for cell in row) + "|")
            print("+---" * self.size + "+")

In [3]:
env = CustomGridEnv()
check_env(env, warn=True)
print("check_env geçti")

obs, info = env.reset(seed=42)
print("Başlangıç gözlemi:", obs)
print("Başlangıç info:", info)

for step in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()
    print(f"adım={step + 1} eylem={action} ödül={reward} terminated={terminated} truncated={truncated} info={info}")
    if terminated or truncated:
        break

check_env geçti
Başlangıç gözlemi: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}
Başlangıç info: {'distance': 4}
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=1 eylem=2 ödül=-0.1 terminated=False truncated=False info={'distance': 4}
+---+---+---+
|   |   |   |
+---+---+---+
| A |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=2 eylem=3 ödül=-0.1 terminated=False truncated=False info={'distance': 3}
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=3 eylem=1 ödül=-0.1 terminated=False truncated=False info={'distance': 4}
+---+---+---+
|   | A |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=4 eylem=0 ödül=-0.1 terminated=False truncated=False info={'distance': 3}
+---+---+---+
|   |   |   |
+---+---+---+
|   | A |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=5 eylem=3 ödül=-0.1 terminated=False truncated

In [4]:
def run(actions):
    env = CustomGridEnv()
    env.reset()
    results = []
    for a in actions:
        obs, reward, terminated, truncated, info = env.step(a)
        results.append((obs["agent"].tolist(), reward, terminated, truncated))
        if terminated or truncated:
            break
    return results

print("Hedefe giden yol :", run([0, 0, 3, 3])[-1])
print("Deliğe giden yol :", run([3, 3, 0])[-1])
print("Duvara çarpma    :", run([2])[-1])
zaman_siniri = run([2] * 60)
print("Zaman sınırı     :", zaman_siniri[-1], "| adım sayısı:", len(zaman_siniri))

Hedefe giden yol : ([2, 2], 10.0, True, False)
Deliğe giden yol : ([2, 1], -10.0, True, False)
Duvara çarpma    : ([0, 0], -0.1, False, False)
Zaman sınırı     : ([0, 0], -0.1, False, True) | adım sayısı: 50


In [5]:
vec_env = make_vec_env(CustomGridEnv, n_envs=1, seed=42)
eval_env = make_vec_env(CustomGridEnv, n_envs=1, seed=0)

untrained = DQN("MultiInputPolicy", vec_env, seed=42)
mean_before, std_before = evaluate_policy(untrained, eval_env, n_eval_episodes=20)
print(f"Eğitimsiz model: {mean_before:.2f} ± {std_before:.2f}")

Eğitimsiz model: -5.00 ± 0.00


In [6]:
model = DQN(
    "MultiInputPolicy",
    vec_env,
    learning_rate=1e-3,
    learning_starts=500,
    exploration_fraction=0.4,
    exploration_final_eps=0.05,
    target_update_interval=250,
    verbose=1,
    seed=42,
)

model.learn(total_timesteps=10_000, progress_bar=True)
model.save("models/dqn_grid_env")

mean_after, std_after = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f"Eğitimli model: {mean_after:.2f} ± {std_after:.2f}")

Using cpu device


Output()

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 9.25     |
|    ep_rew_mean      | -5.83    |
|    exploration_rate | 0.991    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 161      |
|    time_elapsed     | 0        |
|    total_timesteps  | 37       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 13.1     |
|    ep_rew_mean      | -6.21    |
|    exploration_rate | 0.975    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 416      |
|    time_elapsed     | 0        |
|    total_timesteps  | 105      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 15.2     |
|    ep_rew_mean      | -4.75    |
|    exploration_rate | 0.957    |
| time/               |          |
|    episodes       

Eğitimli model: 9.70 ± 0.00


In [7]:
loaded = DQN.load("models/dqn_grid_env")

env = CustomGridEnv()
obs, info = env.reset(seed=0)
total = 0.0
env.render()

for step in range(1, 21):
    action, _ = loaded.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(int(action))
    total += reward
    print(f"adım={step} eylem={int(action)} ödül={reward} terminated={terminated}")
    env.render()
    if terminated or truncated:
        break

print("Toplam ödül:", round(total, 1))

+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=1 eylem=3 ödül=-0.1 terminated=False
+---+---+---+
|   |   |   |
+---+---+---+
| A |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=2 eylem=0 ödül=-0.1 terminated=False
+---+---+---+
|   |   |   |
+---+---+---+
|   | A |   |
+---+---+---+
|   | H | G |
+---+---+---+
adım=3 eylem=0 ödül=-0.1 terminated=False
+---+---+---+
|   |   |   |
+---+---+---+
|   |   | A |
+---+---+---+
|   | H | G |
+---+---+---+
adım=4 eylem=3 ödül=10.0 terminated=True
+---+---+---+
|   |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | A |
+---+---+---+
Toplam ödül: 9.7


In [8]:
arrows = {0: "→", 1: "↑", 2: "←", 3: "↓"}
target = np.array([2, 2], dtype=np.int32)

print("Politika haritası (her hücrede seçilen eylem):")
for r in range(3):
    row = []
    for c in range(3):
        if [r, c] == [2, 1]:
            row.append("H")
        elif [r, c] == [2, 2]:
            row.append("G")
        else:
            obs_rc = {"agent": np.array([r, c], dtype=np.int32), "target": target}
            action, _ = loaded.predict(obs_rc, deterministic=True)
            row.append(arrows[int(action)])
    print(" ".join(row))

Politika haritası (her hücrede seçilen eylem):
↓ → ↓
→ → ↓
↑ H G


## Sonuç

- **Ortam:** `CustomGridEnv`, 3×3 ızgara, koordinatlar (satır, sütun). Ajan sol üstte (0,0) başlar, hedef sağ altta (2,2), delik alt ortada (2,1). Eylemler: 0 = sağ, 1 = yukarı, 2 = sol, 3 = aşağı. Gözlem, ajan ve hedef konumunu içeren bir sözlük (`Dict`). Ödül: hedefe ulaşınca +10, deliğe düşünce -10, diğer her adımda -0.1. Duvara çarpan ajan yerinde kalır (adım cezası yine uygulanır). Bölüm 50 adımda `truncated` ile kesilir.

- **Ortam testi:** `check_env` uyarısız geçti. Rastgele eylemlerle bir bölüm oynatıldı, ardından eylemleri elle yazarak kurallar tek tek doğrulandı: hedefe giden yol `[2, 2]`, ödül 10.0, `terminated=True`. Deliğe giden yol `[2, 1]`, ödül -10.0, `terminated=True`. Duvara çarpma konumu değiştirmedi, ödül -0.1. 60 eylemlik bir dizi 50. adımda `truncated=True` ile bitti.

- **DQN eğitimi:** `make_vec_env` ile sarılan ortamda, gözlem `Dict` olduğu için `MultiInputPolicy` ile DQN eğitildi (`learning_rate=1e-3`, `learning_starts=500`, `exploration_fraction=0.4`, `target_update_interval=250`, 10.000 adım, CPU'da yaklaşık 21 saniye). Varsayılan DQN ayarlarıyla karşılaştırma yapılmadı. Eğitimin sonunda ortalama bölüm uzunluğu yaklaşık 4.2-4.3 adım, ortalama bölüm ödülü yaklaşık 9.5 (keşif adımları dahil).

- **Değerlendirme (20 bölüm, deterministik):** Eğitimsiz model -5.00 ± 0.00 aldı (50 adımlık zaman sınırında aynı hareketi tekrarladı). Eğitimli model 9.70 ± 0.00 aldı, bu ulaşılabilecek en iyi skor (en kısa güvenli yol 4 adım: üç adım -0.1 ve son adım +10).

- **Ajanın davranışı:** Eğitilen model `(0,0)` → `(1,0)` → `(1,1)` → `(1,2)` → `(2,2)` yolunu izledi, 4 adım, toplam ödül 9.7. Politika haritası (her hücrede seçilen eylem):
  - `↓ → ↓`
  - `→ → ↓`
  - `↑ H G`

  Elle kontrol edildiğinde yürünebilir yedi hücrenin hepsinde seçilen eylem hedefe en kısa güvenli yolu veriyor. Ajan, eğitimde nadir ziyaret edilen (2,0) hücresinde bile deliğe giden sağ yerine yukarı hareketini seçmiş.

- **Sınırlılıklar:** Ortam ve politika deterministik olduğu için ± 0.00, 20 bağımsız deneme anlamına gelmiyor. Tek tohum ve tek eğitim çalıştırması kullanıldı. Gözlemde deliğin konumu yok, ajan bunu eğitimle öğrenmiş, delik taşınırsa modelin uyum sağlayıp sağlamayacağı denenmedi. 3×3'lük küçük bir ortam, DQN'in kapasitesini zorlamıyor.

- **Genel değerlendirme:** Gymnasium'un `Env` yapısı (`reset`, `step`, gözlem ve eylem uzayları, `terminated` ve `truncated` ayrımı) kendi ortamımızda uygulandı ve Stable Baselines3 ile uyumlu çalıştığı `check_env` ve DQN eğitimiyle doğrulandı. Ödül yapısı ajanın hem deliği kaçınmasını hem en kısa yolu bulmasını sağladı.